# Protobuf Basics — 07: Protobuf vs JSON vs Avro

Comprehensive comparison across all dimensions to guide format selection.

Topics: size benchmark, serialization speed, schema evolution, tooling,
ecosystem support, decision guide.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Size Comparison



In [ ]:

import json as pyjson, struct as st, random, time as tm

random.seed(42)
N = 5_000
CATS=["Electronics","Books","Clothing","Food","Sports"]
CTRS=["US","UK","DE","FR","JP"]

orders = [{"order_id":i,"customer_id":random.randint(1,500),
           "product":f"Product_{random.randint(1,50)}","category":random.choice(CATS),
           "country":random.choice(CTRS),"quantity":random.randint(1,10),
           "price":round(random.uniform(5,500),2),"revenue":0.0,
           "status":random.choice(["pending","confirmed","shipped"])} for i in range(N)]
for o in orders: o["revenue"]=round(o["price"]*o["quantity"],2)

# JSON size
def enc_v(v):
    r=b""
    while v>0x7F: r+=bytes([(v&0x7F)|0x80]); v>>=7
    return r+bytes([v&0x7F])
def enc_f(fn,wt,val): return enc_v((fn<<3)|wt)+val
def enc_s(fn,s): e=s.encode(); return enc_f(fn,2,enc_v(len(e))+e)
def enc_i(fn,n): return enc_f(fn,0,enc_v(n))
def enc_d(fn,d): return enc_f(fn,1,st.pack('<d',d))

t0=tm.time()
json_bytes=[pyjson.dumps(o).encode() for o in orders]; t_json=tm.time()-t0
json_total=sum(len(b) for b in json_bytes)

t0=tm.time()
proto_bytes=[enc_i(1,o["order_id"])+enc_i(2,o["customer_id"])+enc_s(3,o["product"])+
             enc_s(4,o["category"])+enc_s(5,o["country"])+enc_i(6,o["quantity"])+
             enc_d(7,o["price"])+enc_d(8,o["revenue"])+enc_s(9,o["status"])
             for o in orders]; t_proto=tm.time()-t0
proto_total=sum(len(b) for b in proto_bytes)

print(f"Serialization of {N:,} orders:")
print(f"  {'Format':<12} {'Total KB':>10} {'Avg bytes':>12} {'Serialize':>12}")
print(f"  {'-'*50}")
print(f"  {'JSON':<12} {json_total/1024:>8.0f} KB {json_total/N:>10.0f} B {t_json:>10.3f}s")
print(f"  {'Protobuf':<12} {proto_total/1024:>8.0f} KB {proto_total/N:>10.0f} B {t_proto:>10.3f}s")
print(f"  Protobuf vs JSON: {json_total/proto_total:.1f}x smaller, {t_json/t_proto:.1f}x faster")


## Step 2 — Schema Evolution Comparison



In [ ]:

print("""
Schema evolution head-to-head:

Feature                  Protobuf         Avro              JSON
──────────────────────────────────────────────────────────────────
Schema language          .proto           JSON Schema       None
Schema in data           No               Yes (header)      Yes (self)
Schema registry          ✅ Confluent SR  ✅ Confluent SR   ⚠️ Optional
Add field                ✅ Free          ✅ With default    ✅ No contract
Remove field             ✅ Reserve #     ✅ With default    ✅ No contract
Rename field             ✅ Field # stays ⚠️ Needs alias    ✅ Change name
Change type (compatible) ✅ Wire compat   ✅ Resolution      ✅ No check
Change type (incompat)   ❌ Corruption    ❌ Rejected        ❌ Silent corruption
Compatibility check      Manual           Automatic (SR)    None

Winner for evolution: Avro (automatic compatibility checking in Schema Registry)
Winner for flexibility: Protobuf (rename fields freely, strong binary evolution)
""")


## Step 3 — Ecosystem and Tooling



In [ ]:

print("""
Ecosystem comparison:

  Protobuf:
    gRPC: ✅ Native (designed for it)
    Kafka: ✅ Common (Confluent Schema Registry support)
    Spark: ✅ Built-in (Spark 3.4+)
    Flink: ✅ Supported
    Python: ✅ protobuf library
    REST APIs: ❌ (use JSON or gRPC-gateway)
    Human-readable: ❌
    Hive/Presto: ❌ Not supported

  Avro:
    gRPC: ❌
    Kafka: ✅ Standard (Confluent Schema Registry native)
    Spark: ✅ (with spark-avro JAR)
    Flink: ✅
    Python: ✅ fastavro
    REST APIs: ❌
    Human-readable: Schema (JSON), Data (binary)

  JSON:
    gRPC: ❌ (REST instead)
    Kafka: ✅ Common (no schema enforcement)
    Spark: ✅ Built-in
    REST APIs: ✅ Standard
    Human-readable: ✅
    Tooling: ✅ jq, any editor, browser
    Schema: ❌ Optional (JSON Schema, not enforced)
""")


## Step 4 — Decision Guide



In [ ]:

print("""
Format selection guide:

  Use Protobuf when:
    ✅ Building gRPC services
    ✅ Kafka pipelines where message size is critical (IoT, high-throughput)
    ✅ Multi-language teams (Java, Go, Python, C++) need same schema
    ✅ You want the smallest possible binary messages
    ✅ Your team is comfortable maintaining .proto files

  Use Avro when:
    ✅ Kafka pipelines with Confluent Schema Registry
    ✅ Need automatic schema compatibility checking
    ✅ Hadoop/Hive ecosystem
    ✅ Schema evolution is frequent
    ✅ Landing zone for Spark batch processing

  Use JSON when:
    ✅ REST API payloads
    ✅ Configuration files
    ✅ Human-readable logging
    ✅ Quick prototyping
    ✅ No schema contract needed

  Analytics storage: use Parquet (not any of the above)
  
  Most common production pattern:
    Source → Protobuf/Avro (Kafka) → Spark → Parquet/Delta (analytics)
""")


## Summary



In [ ]:

print("""
Format comparison summary:
  Size:    Protobuf < Avro < JSON
  Speed:   Protobuf ≈ Avro > JSON
  gRPC:    Protobuf only
  Kafka:   All (Avro/Protobuf preferred for schema enforcement)
  Spark:   All (Parquet preferred for analytics)
  Human:   JSON only
  Evolution: Avro (automatic) ≥ Protobuf (manual) > JSON (none)
""")
